In [14]:
df = pd.read_parquet("data/clustered_data.parquet")

print("Cluster 0 date range:", df[df.cluster == 0]["date"].min(), "to", df[df.cluster == 0]["date"].max())
print("Cluster 1 date range:", df[df.cluster == 1]["date"].min(), "to", df[df.cluster == 1]["date"].max())
print("Cluster 2 date range:", df[df.cluster == 2]["date"].min(), "to", df[df.cluster == 2]["date"].max())

df["year"] = pd.to_datetime(df["date"]).dt.year
print(df.groupby(["year", "cluster"]).size().unstack(fill_value=0))

Cluster 0 date range: 2016-06-29 00:00:00 to 2021-07-01 00:00:00
Cluster 1 date range: 2016-06-29 00:00:00 to 2020-12-31 00:00:00
Cluster 2 date range: 2016-06-29 00:00:00 to 2021-02-01 00:00:00
cluster       0      1       2
year                          
2016      38017    773   39960
2017     110227     24   44770
2018      61191  22261   74666
2019      22083  30732  101484
2020      19132  89815   47857
2021      76734      0    1430


In [15]:
import pandas as pd

df = pd.read_parquet("data/clustered_data_depth2.parquet")
top_cluster = df[df["cluster"] == 2]

print(f"Rows: {len(top_cluster)}, unique tickers: {top_cluster['ticker'].nunique()}")
print(f"Rows per ticker (avg): {len(top_cluster) / top_cluster['ticker'].nunique():.1f}")
print()
print("Top 15 tickers by row count in this cluster:")
print(top_cluster["ticker"].value_counts().head(15))

print()
print("What % of the cluster do the top 10 tickers represent?")
top10_count = top_cluster["ticker"].value_counts().head(10).sum()
print(f"{top10_count} / {len(top_cluster)} = {top10_count/len(top_cluster):.1%}")

Rows: 911, unique tickers: 136
Rows per ticker (avg): 6.7

Top 15 tickers by row count in this cluster:
ticker
GPRK    7
SPND    7
FET     7
NGS     7
GLNG    7
TK      7
UGP     7
OVV     7
TDW     7
PBT     7
BP      7
EFXT    7
REPX    7
EQT     7
CNX     7
Name: count, dtype: int64

What % of the cluster do the top 10 tickers represent?
70 / 911 = 7.7%


In [16]:
df = pd.read_parquet("data/clustered_data_depth2.parquet")
top_cluster = df[df["cluster"] == 2]

print("Sector breakdown:")
print(top_cluster["sector"].value_counts())

print("\nDate range and distribution:")
print(top_cluster["date"].agg(["min", "max"]))
top_cluster["year"] = pd.to_datetime(top_cluster["date"]).dt.year
print(top_cluster["year"].value_counts().sort_index())

Sector breakdown:
sector
Energy    911
Name: count, dtype: int64

Date range and distribution:
min   2020-03-11
max   2020-06-26
Name: date, dtype: datetime64[ms]
year
2020    911
Name: count, dtype: int64


In [17]:
import json

with open("data/company_tickers.json") as f:
    data = json.load(f)

symbols = {stock["ticker"] for stock in data.values()}

watchlist = ["WDC", "SNDK", "MU", "STX"]  # Western Digital, Sandisk, Micron, Seagate
for t in watchlist:
    print(f"{t}: {'present' if t in symbols else 'MISSING'}")

WDC: present
SNDK: present
MU: present
STX: present


In [19]:
import pandas as pd

df = pd.read_parquet("data/data.parquet")

watchlist = ["WDC", "SNDK", "MU", "STX"]

for t in watchlist:
    count = (df["ticker"] == t).sum()
    if count > 0:
        dates = df[df["ticker"] == t]["date"]
        print(f"{t}: {count} rows ({dates.min()} to {dates.max()})")
    else:
        print(f"{t}: NOT FOUND in data.parquet")

WDC: 253 rows (2016-06-29 00:00:00 to 2021-07-01 00:00:00)
SNDK: NOT FOUND in data.parquet
MU: 253 rows (2016-06-29 00:00:00 to 2021-07-01 00:00:00)
STX: 253 rows (2016-06-29 00:00:00 to 2021-07-01 00:00:00)


In [1]:
import pandas as pd
import json

TICKER = "SNDK"

# Stage 1: company_tickers.json
print("=" * 60)
print("STAGE 1: company_tickers.json")
print("=" * 60)
with open("data/company_tickers.json") as f:
    data = json.load(f)
symbols = {stock["ticker"] for stock in data.values()}
print(f"{TICKER} in symbol list: {TICKER in symbols}")

# Stage 2: raw_data.parquet (post-download)
print("\n" + "=" * 60)
print("STAGE 2: raw_data.parquet")
print("=" * 60)
raw = pd.read_parquet("data/raw_data.parquet")
close = raw["close"]
volume = raw["volume"]

if TICKER in close.columns:
    series = close[TICKER].dropna()
    vol = volume[TICKER].dropna() if TICKER in volume.columns else None
    print(f"{TICKER} found in raw close data")
    print(f"  Rows: {len(series)}  (MIN_ROWS threshold is 3574)")
    print(f"  Date range: {series.index.min()} to {series.index.max()}")
    print(f"  Price range: {series.min():.2f} to {series.max():.2f}")
    if vol is not None:
        print(f"  Volume data present: {len(vol)} rows")
    else:
        print("  Volume data: MISSING")
else:
    print(f"{TICKER} NOT FOUND in raw_data.parquet — dropped during download")
    print("  (likely: rate-limited, batch failure, or didn't meet MIN_ROWS during get_raw_data.py)")

# Stage 3: raw_info.parquet (sector/shares/PE lookup)
print("\n" + "=" * 60)
print("STAGE 3: raw_info.parquet")
print("=" * 60)
info = pd.read_parquet("data/raw_info.parquet")
row = info[info["ticker"] == TICKER]
if not row.empty:
    print(row.to_string(index=False))
else:
    print(f"{TICKER} NOT FOUND in raw_info.parquet")

# Stage 4: data.parquet (final output)
print("\n" + "=" * 60)
print("STAGE 4: data.parquet")
print("=" * 60)
final = pd.read_parquet("data/data.parquet")
count = (final["ticker"] == TICKER).sum()
print(f"{TICKER} rows in final dataset: {count}")

STAGE 1: company_tickers.json
SNDK in symbol list: True

STAGE 2: raw_data.parquet
SNDK NOT FOUND in raw_data.parquet — dropped during download
  (likely: rate-limited, batch failure, or didn't meet MIN_ROWS during get_raw_data.py)

STAGE 3: raw_info.parquet
SNDK NOT FOUND in raw_info.parquet

STAGE 4: data.parquet
SNDK rows in final dataset: 0
